# EEG Preprocessing — Brain Invaders (bi2015a)

Preprocessing pipeline for the **bi2015a** dataset (Brain Invaders, P300-based BCI).  
Adapted from our professor's MNE preprocessing notebook.

**Reference:** Congedo et al. (2017) — *A new generation of Brain-Computer Interface based on Riemannian geometry*

### Protocol overview
Participants focus on a target face in a 6×6 grid. Faces flash one at a time at random. The brain produces a **P300 response** (~300 ms post-stimulus) only when the attended (target) face flashes. A classifier trained on these EEG responses can decode which face the user was looking at — this is the BCI.

| Column | Meaning |
|--------|---------|
| `Trigger` | `1` = stimulus onset (a face flashed), `0` = nothing |
| `Target`  | `1` = that flash was the target face, `0` = non-target |

**Sampling rate:** 512 Hz &nbsp;|&nbsp; **Channels:** 32 EEG &nbsp;|&nbsp; **Target : non-target ratio:** 1 : 5 per repetition

> ⚠️ **Data quality note:** Subjects **1, 27, 44–50** are flagged in the paper for power-line interference or corrupted recordings. They are loaded normally here — inspect the PSD and epoch rejection rates carefully and decide whether to include them in your analysis.

## 1. Imports

This cell loads all Python libraries the notebook needs. Here is what each one does in this context:

| Library | Alias | Purpose in this notebook |
|---------|-------|--------------------------|
| `numpy` | `np` | Fast numerical arrays; used to reshape and scale EEG data |
| `pandas` | `pd` | Loads CSV files and stores per-epoch metadata (target/non-target labels) |
| `matplotlib.pyplot` | `plt` | Draws all figures |
| `mne` | — | The core EEG/MEG analysis library: filtering, epoching, visualisation, and saving `.fif` files |
| `pathlib.Path` | — | Builds file paths in a way that works on Windows, Mac, and Linux without manual string concatenation |

`%matplotlib inline` tells Jupyter to render figures inside the notebook rather than opening a separate window.

`mne.set_log_level('WARNING')` silences MNE's verbose internal messages so the output stays readable. Change it to `'INFO'` if a step fails and you want a detailed trace of what MNE is doing.

In [ ]:
%matplotlib inline                   # render figures inside the notebook, not a separate window
import numpy as np                   # numerical arrays and maths
import pandas as pd                  # tabular data — CSV loading and epoch metadata
import matplotlib.pyplot as plt      # all plotting
import mne                           # EEG/MEG analysis: filtering, epoching, visualisation, file I/O
from pathlib import Path             # cross-platform file path construction

mne.set_log_level('WARNING')  # suppress MNE's verbose progress messages; change to 'INFO' to debug

## 2. Configuration

**Change `SUBJECT` and `SESSION` here before running the notebook.** All file paths and labels are built from these two numbers automatically — you never need to edit a path manually.

| Variable | What it controls |
|----------|-----------------|
| `SUBJECT` | Which participant's data to load (integers 1–43). Subjects 1 and 27 are flagged as bad recordings in the bi2015a paper — avoid them. |
| `SESSION` | Which recording session: 1 (slow flashes, 110 ms), 2 (medium, 80 ms), or 3 (fast, 50 ms) |

**Why three sessions?**  
The bi2015a study tested whether stimulus duration affects P300 decoding accuracy. Each participant completed three identical sessions at different flash speeds. You can process one session at a time to compare results, or batch-process all three and pool the epochs.

The `:02d` format specifier inside f-strings zero-pads single-digit numbers (e.g. subject 3 → `"03"`), matching the zero-padded filenames in the dataset.

`OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)` creates the output folder and any missing parent folders the first time the notebook runs; it does nothing if the folder already exists.

In [ ]:
# ── USER CONFIG ──────────────────────────────────────────────────────────────
SUBJECT     = 1   # participant number (1–43); avoid 1 and 27 (bad recordings)
SESSION     = 1   # recording session: 1 (slow flashes), 2 (medium), 3 (fast)

# :02d zero-pads single-digit numbers so subject 3 → "03", matching the filenames
_PROJECT_ROOT = Path().resolve()
DATA_ROOT   = _PROJECT_ROOT / "data" / "raw" / f"subject_{SUBJECT:02d}_csv"
OUTPUT_ROOT = _PROJECT_ROOT / "data" / "preprocessed"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)   # create folder if it doesn't exist yet

# Flash duration per session (ms) — from Table 2 of the bi2015a paper.
# Slower flashes give the brain more time to respond; useful context when
# comparing classifier performance across sessions.
FLASH_DURATION_MS = {
    1: 110,   # slowest: 110 ms on-screen
    2: 80,
    3: 50,    # fastest: 50 ms on-screen
}

# ── DERIVED ──────────────────────────────────────────────────────────────────
# Path / "string" is Python's way to join path components — works on all OSes
csv_path = DATA_ROOT   / f"subject_{SUBJECT:02d}_session_{SESSION:02d}.csv"
fif_path = OUTPUT_ROOT / f"subject_{SUBJECT:02d}_session_{SESSION:02d}_epo.fif"
# _epo.fif is MNE's conventional suffix for epoch files

flash_ms = FLASH_DURATION_MS[SESSION]
print(f"Subject  : {SUBJECT}")
print(f"Session  : {SESSION}  (flash duration = {flash_ms} ms)")
print(f"Input    : {csv_path}")
print(f"Output   : {fif_path}")
print(f"Flash duration : {flash_ms} ms  (Session 1=110ms, Session 2=80ms, Session 3=50ms)")

## 3. Load the CSV

The bi2015a CSV files have **no header row** — the first row is already data. We supply the 35 column names manually based on the dataset documentation.

The `Trigger` and `Target` columns are stored as floats (`0.0` / `1.0`) in some sessions due to how the files were written; we cast them to `int` for clarity.

In [ ]:
COLUMNS = [
    "Time",
    "FP1", "FP2", "AFz", "F7",  "F3",  "F4",  "F8",
    "FC5", "FC1", "FC2", "FC6", "T7",  "C3",  "Cz",  "C4",  "T8",
    "CP5", "CP1", "CP2", "CP6", "P7",  "P3",  "Pz",  "P4",  "P8",
    "PO7", "O1",  "Oz",  "O2",  "PO8", "PO9", "PO10",
    "Trigger", "Target",
]
# Electrode names use the 10-05 international system:
# F=frontal, C=central, P=parietal, O=occipital, T=temporal
# z=midline, odd number=left hemisphere, even number=right hemisphere

EEG_CHANNELS = COLUMNS[1:33]   # skip "Time" (index 0) and "Trigger","Target" (indices 33, 34)
SFREQ        = 512.0            # amplifier sampling rate: 512 samples recorded per second

df = pd.read_csv(csv_path, header=None, names=COLUMNS)
# header=None: no column header row in the file — the first row is already EEG data
# names=COLUMNS: we supply the column names ourselves, in the order they appear in the file

# Some sessions store these columns as floats (e.g. "1.0" instead of "1")
df["Trigger"] = df["Trigger"].astype(int)
df["Target"]  = df["Target"].astype(int)

print(f"Shape    : {df.shape[0]:,} samples × {df.shape[1]} columns")
print(f"Duration : {df['Time'].max():.1f} s  ({df['Time'].max()/60:.1f} min)")
print(f"Trigger=1 events : {(df['Trigger']==1).sum()}")
print(f"Target=1  events : {(df['Target']==1).sum()}")
df.head(3)

## 4. Build an MNE `RawArray`

MNE works with data in **Volts**. The bi2015a data is stored in **µV**, so we divide by `1e6`.

We add a dedicated **STIM channel** (`STI014`) built from the `Trigger` column. MNE uses this channel to detect events — it must stay in counts (do **not** scale it).

In [ ]:
# EEG data: select the 32 electrode columns, transpose to (channels, samples),
# and convert µV → V by dividing by 1,000,000 — MNE works in Volts internally
eeg_data  = df[EEG_CHANNELS].values.T / 1e6
# .values returns a numpy array of shape (n_samples, 32)
# .T transposes to (32, n_samples) because MNE expects rows=channels, columns=time

# STIM channel: the Trigger column holds 0 everywhere except stimulus onset (value 1)
# np.newaxis adds a dimension: (n_samples,) → (1, n_samples) so it stacks with eeg_data
stim_data = df["Trigger"].values[np.newaxis, :]

# Stack EEG (32 rows) + STIM (1 row) into a single (33, n_samples) array
raw_data  = np.vstack([eeg_data, stim_data])

ch_names = EEG_CHANNELS + ["STI014"]                   # "STI014" is MNE's conventional stimulus channel name
ch_types = ["eeg"] * len(EEG_CHANNELS) + ["stim"]      # "stim" tells MNE not to apply EEG filters to this channel

# create_info bundles channel names, types, and sampling rate into MNE's metadata container
info = mne.create_info(ch_names=ch_names, sfreq=SFREQ, ch_types=ch_types)

# RawArray wraps our numpy array + metadata into a full MNE Raw object,
# unlocking MNE's filter, epoch, and visualisation methods
raw  = mne.io.RawArray(raw_data, info, verbose=False)

print(raw)

### Visualisation: Raw Signal

Use the cell below to visually inspect the raw EEG traces (interactive — requires a GUI backend). What to look for:

- **Large slow drifts** (< 1 Hz wandering baseline) — a clear sign the high-pass filter is needed
- **Obvious bad channels** — a completely flat line or extreme noise throughout the recording; note them for later
- **Signal scale** — bi2015a data is in µV; typical scalp EEG is 10–100 µV; values much larger suggest amplifier saturation

In [5]:
# Interactive — requires a GUI backend (e.g. Qt5Agg). Commented out for shared notebooks.
# raw.plot(duration=5, n_channels=10, scalings='auto')

## 5. Montage

A **montage** is a map that tells MNE the 3-D position of each electrode on the scalp. Without it, MNE knows the channel names but not where they physically sit on the head, so topographic plots and source-level analysis are impossible.

The Brain Invaders setup uses the **10-05 international system** — a widely agreed-upon grid of electrode positions covering the scalp. The naming convention is:
- **Letter prefix** indicates the brain region: F=frontal, C=central, P=parietal, O=occipital, T=temporal, FC=fronto-central, etc.
- **Number or z suffix**: odd = left hemisphere, even = right hemisphere, z = midline (e.g. Cz is the very top-centre of the head)

We use the pre-built `standard_1005` montage (an atlas of 343 standard positions) rather than digitising electrode locations individually. This is an approximation, but accurate enough for topographic visualisation and for comparing results across studies.

`on_missing="warn"` prints a warning for any channel whose name is not in the standard atlas (e.g. PO9/PO10 are non-standard extensions) but does not crash.

In [ ]:
montage = mne.channels.make_standard_montage("standard_1005")  # load the standard 10-05 electrode position atlas
raw.set_montage(montage, match_case=False, on_missing="warn")   # attach positions to the Raw object; warn on unknown names
print("Montage applied.")

## 6. Visualisation: Raw PSD

Plot the power spectral density of the **unfiltered** raw signal. What to look for:

- **50 Hz peak** — a sharp spike at 50 Hz indicates European power-line interference; it should disappear after the notch filter
- **1/f shape** — EEG power naturally decreases with frequency. A heavily distorted slope or flat spectrum can indicate amplifier saturation or bad channels
- **Channel spread** — `average=False` plots each channel individually so outliers are visible

In [ ]:
spectrum = raw.compute_psd(fmax=60)   # compute power at each frequency from 0 to 60 Hz

# Convert power units for readability:
# spectrum.get_data() returns power in V²/Hz (very small numbers like 1e-12)
# × 1e12 converts V² → µV² (the scale humans are used to seeing for EEG)
# 10 * log10(...) converts linear power to decibels — a log scale that
# compresses the wide range of EEG power into a readable ~60 dB window
psds_db  = 10 * np.log10(spectrum.get_data() * 1e12)  # V²/Hz → dB µV²/Hz

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(spectrum.freqs, psds_db.T, alpha=0.35, color='steelblue', linewidth=0.5)
# psds_db shape is (n_channels, n_freqs); .T transposes so each column = 1 channel,
# and matplotlib draws each column as a separate thin line
ax.plot(spectrum.freqs, psds_db.mean(axis=0), color='black', linewidth=2)  # channel-average in bold
ax.axvline(50, color='red', linestyle='--', alpha=0.8, label='50 Hz (power line)')
ax.set_xlabel('Frequency (Hz)')
ax.set_ylabel('Power (dB µV²/Hz)')
ax.set_title('PSD — Raw signal (each line = 1 channel)')
ax.legend()
plt.tight_layout()
plt.show()

## 7. Filtering

Two filters are applied in sequence:

1. **Notch filter at 50 Hz** — removes power-line interference (European electrical grid).
2. **Bandpass 0.1–30 Hz** — removes slow drift (below 0.1 Hz) and high-frequency noise above 30 Hz. The P300 component lives in the 1–10 Hz range, so nothing physiologically relevant is lost.

In [ ]:
raw_notched = raw.copy().notch_filter(freqs=50, picks="eeg")
# .copy() creates an independent copy so the original `raw` object is preserved unchanged
# picks="eeg" applies the filter only to EEG channels — the STIM channel must stay unfiltered
print("Notch filter at 50 Hz applied.")

### Visualisation: PSD after Notch Filter

The **50 Hz peak should now be gone**. Compare this plot to the raw PSD above — everything else should look identical. If the peak is still visible, the notch filter may not have been applied to all channels.

In [ ]:
spectrum = raw_notched.compute_psd(fmax=60)   # recompute PSD on the notch-filtered signal
psds_db  = 10 * np.log10(spectrum.get_data() * 1e12)  # V²/Hz → dB µV²/Hz (same conversion as before)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(spectrum.freqs, psds_db.T, alpha=0.35, color='steelblue', linewidth=0.5)
ax.plot(spectrum.freqs, psds_db.mean(axis=0), color='black', linewidth=2)
ax.axvline(50, color='red', linestyle='--', alpha=0.8, label='50 Hz (should be flat now)')
ax.set_xlabel('Frequency (Hz)')
ax.set_ylabel('Power (dB µV²/Hz)')
ax.set_title('PSD — After notch filter at 50 Hz')
ax.legend()
plt.tight_layout()
plt.show()

### Step 2: Bandpass filter (0.1–30 Hz)

The notch filter only removed the narrow 50 Hz spike. We now apply a **bandpass filter** to cut everything outside the 0.1–30 Hz range.

- **Low-cut at 0.1 Hz (high-pass)** — removes very slow voltage drifts caused by electrode sweat, body movement, and DC offsets. These drifts can be tens of µV and would corrupt the baseline correction we apply during epoching.
- **High-cut at 30 Hz (low-pass)** — removes high-frequency muscle noise (EMG) from jaw clenching or scalp tension, and any residual electrical noise. Brain signals measured at the scalp for P300-based BCIs carry essentially no useful information above 30 Hz.

The P300 component peaks around 300 ms post-stimulus and is dominated by frequencies between 1–10 Hz — well within the 0.1–30 Hz passband, so nothing physiologically relevant is discarded.

`.copy()` makes a new copy so `raw_notched` stays unchanged and we can compare before/after in the plot below.

In [ ]:
raw_filtered = raw_notched.copy().filter(l_freq=0.1, h_freq=30, picks="eeg")
# l_freq=0.1: high-pass edge — removes anything below 0.1 Hz (slow electrode drift)
# h_freq=30:  low-pass edge  — removes anything above 30 Hz (muscle noise, EMG)
# picks="eeg": apply only to EEG channels, not the STIM channel
print("Bandpass filter 0.1–30 Hz applied.")

### Visualisation: PSD after Bandpass Filter

The signal should now **drop sharply below 0.1 Hz and above 30 Hz**. The retained 0.1–30 Hz band contains all P300-relevant activity — the N200, P300 and late positive components all live below 10 Hz.

In [ ]:
spectrum = raw_filtered.compute_psd(fmax=60)   # recompute PSD on the fully filtered signal
psds_db  = 10 * np.log10(spectrum.get_data() * 1e12)  # V²/Hz → dB µV²/Hz

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(spectrum.freqs, psds_db.T, alpha=0.35, color='steelblue', linewidth=0.5)
ax.plot(spectrum.freqs, psds_db.mean(axis=0), color='black', linewidth=2)
ax.axvspan(0, 0.1, alpha=0.12, color='red', label='Filtered out (< 0.1 Hz)')   # axvspan = shaded vertical band
ax.axvspan(30, 60, alpha=0.12, color='red', label='Filtered out (> 30 Hz)')
ax.set_xlabel('Frequency (Hz)')
ax.set_ylabel('Power (dB µV²/Hz)')
ax.set_title('PSD — After bandpass filter (0.1–30 Hz)')
ax.legend()
plt.tight_layout()
plt.show()

## 8. Event detection & Epoching

MNE scans the STIM channel for rising edges to locate stimulus onsets. Each detected event becomes the centre of a 1.2-second epoch:

| Parameter | Value | Rationale |
|-----------|-------|-----------|
| `tmin` | −0.2 s | 200 ms pre-stimulus baseline |
| `tmax` | 1.0 s | captures P300 (300–600 ms) and late slow-wave components up to 1 s |
| Baseline | (−0.2, 0) | subtract mean of pre-stimulus window |
| Reject threshold | 100 µV peak-to-peak | simple artefact rejection; no ICA |

In [ ]:
events = mne.find_events(raw_filtered, stim_channel="STI014", verbose=False)
# events is a numpy array of shape (n_events, 3):
#   column 0 — sample index in the recording where the trigger occurred
#   column 1 — always 0 for this dataset (used for "previous event ID" in some paradigms)
#   column 2 — the trigger value (1 = stimulus onset for bi2015a)
print(f"Events found : {len(events)}")
print(f"Event IDs    : {np.unique(events[:, 2])}")  # events[:, 2] = third column = trigger values

### Visualisation: Event Timing

Plot stimulus onsets over time. What to check:

- **Event count** should match `Trigger=1` from Section 3
- **Even spacing** — events should be distributed throughout the recording with no large unexplained gaps (gaps may indicate dropped triggers or a pause in the experiment)

In [ ]:
mne.viz.plot_events(
    events,
    sfreq=raw_filtered.info['sfreq'],       # sampling rate: needed to convert sample numbers → seconds on the x-axis
    first_samp=raw_filtered.first_samp,     # MNE's internal sample offset (usually 0 for RawArray; needed for file-based data)
)

### Cutting the continuous signal into epochs

`mne.find_events` gave us the sample index of every stimulus flash. Now we tell MNE to cut a short segment (an **epoch**) of the filtered EEG around each of those timestamps.

Think of it like slicing the 20-minute recording into hundreds of 1.2-second clips — one clip per flash — and stacking them into a 3-D array of shape `(n_epochs, n_channels, n_timepoints)`.

**Key parameters explained:**

| Parameter | Value | What it does |
|-----------|-------|--------------|
| `tmin=-0.2` | −200 ms | Epoch starts 200 ms *before* the flash — this is the **pre-stimulus baseline** period |
| `tmax=1.0` | +1000 ms | Epoch ends 1000 ms *after* the flash — enough to see the P300 peak (~300–500 ms) and later components |
| `baseline=(-0.2, 0)` | pre-stimulus window | **Baseline correction**: subtracts the mean voltage during −200→0 ms from every timepoint. This removes any slow voltage offset that happened to be present before the flash, so all epochs start from a common zero and trial-to-trial differences reflect only brain *responses* |
| `reject={"eeg": 100e-6}` | 100 µV | **Artefact rejection**: discards any epoch in which *any* channel exceeds ±100 µV. Genuine brain signals are 5–20 µV; eye blinks produce 100–300 µV spikes. This threshold removes the most contaminated trials without ICA. `100e-6` is 100 µV expressed in Volts (MNE's internal unit) |
| `preload=True` | — | Loads all epoch data into RAM immediately; required for attaching metadata and saving |

In [ ]:
epochs = mne.Epochs(
    raw_filtered,
    events,
    event_id={"stimulus": 1},  # only cut epochs around trigger value 1 (all flashes)
    tmin=-0.2,                  # start each epoch 200 ms BEFORE the flash (pre-stimulus baseline)
    tmax=1.0,                   # end each epoch 1000 ms AFTER the flash (captures P300 + late components)
    baseline=(-0.2, 0),         # subtract mean of the −200 to 0 ms window from every timepoint
                                # this removes any DC offset present before the flash
    reject={"eeg": 100e-6},     # discard epochs where any EEG channel exceeds ±100 µV
                                # 100e-6 = 100 µV in Volts (MNE's internal unit)
    preload=True,               # load all epoch data into RAM now (required for metadata and save)
    verbose=False,
)

print(epochs)

### Visualisation: Epoch Image at Cz

Each horizontal stripe is one epoch (time on x-axis, amplitude colour-coded). A consistent warm-coloured band around 300–500 ms in target epochs indicates a P300. Channel Cz is a central midline electrode where P300 is typically prominent.

**Interactive — requires a GUI backend. Commented out for shared notebooks.**

In [15]:
# Interactive — requires a GUI backend. Commented out for shared notebooks.
# epochs.plot_image(picks=['Cz'], combine=None)

## 9. Extract labels

The `Target` column records whether each stimulus was a target (1) or non-target (0). We align these labels to the trigger sample indices returned by `mne.find_events`, then keep only the labels for epochs that survived artefact rejection.

`epochs.selection` contains the indices (into the original events array) of the epochs that were kept — this is what we use to index into our label array.

In [ ]:
# events[:, 0] contains the sample index of each flash in the raw recording.
# Subtracting raw_filtered.first_samp converts MNE's internal sample index
# into a row index into our original DataFrame, so we can look up the Target label.
labels      = df["Target"].values[events[:, 0] - raw_filtered.first_samp]

# epochs.selection holds the indices (into the original events array) of every epoch
# that survived artefact rejection. Use it to keep only the labels for surviving epochs.
labels_kept = labels[epochs.selection]

n_target    = int(labels_kept.sum())                # sum works because target=1, non-target=0
n_nontarget = len(labels_kept) - n_target

print(f"Epochs after rejection : {len(labels_kept)}  "
      f"(dropped {len(events) - len(labels_kept)})")
print(f"  Target     (1) : {n_target}")
print(f"  Non-target (0) : {n_nontarget}")
if n_target > 0:
    print(f"  Ratio          : 1 : {n_nontarget / n_target:.1f}")

### Attaching labels to the epoch object

We now have two separate things that belong together:
1. The cleaned EEG data for each surviving epoch (inside the `epochs` object)
2. A target / non-target label for each epoch (`labels_kept`)

`epochs.metadata` is a pandas DataFrame that MNE keeps permanently synchronised with the epoch data. Storing the labels here means:
- The label travels with the epoch in the `.fif` file — when you `mne.read_epochs(fif_path)` in a later notebook, `epochs.metadata["target"]` is immediately available
- You can filter epochs by label with simple boolean indexing: `epochs[epochs.metadata["target"] == 1]`

Without this step, a downstream notebook would have to re-parse the original CSV and repeat the label-alignment logic every time it needed labels.

In [ ]:
# epochs.metadata is a pandas DataFrame with one row per surviving epoch.
# Assigning it here permanently links the labels to the epoch data — when we
# save and later reload the .fif file, the metadata comes with it automatically.
epochs.metadata = pd.DataFrame(
    {"target": labels_kept},
    index=range(len(labels_kept)),
)

# value_counts() counts occurrences of each unique value; .rename() gives them readable names
print(epochs.metadata["target"].value_counts()
      .rename(index={0: "non-target", 1: "target"})
      .to_string())

### Visualisation: ERP at Cz — Target vs Non-target

This is the **key sanity check** for a P300 dataset. The target ERP should show a clear positive peak (**P300**) between 300–600 ms post-stimulus that the non-target ERP does not. If both curves are flat and indistinguishable, it likely indicates a label alignment problem or a noisy recording.

Channel Cz is a central midline electrode where the P300 is typically maximal in the Brain Invaders paradigm.

In [ ]:
# Filter the epoch object by metadata label
target_epochs    = epochs[epochs.metadata['target'] == 1]
nontarget_epochs = epochs[epochs.metadata['target'] == 0]

fig, ax = plt.subplots(figsize=(8, 4))

# Average all target epochs at channel Cz.
# Individual trials are too noisy to see the ERP; averaging hundreds of trials
# cancels out random neural activity, leaving only the time-locked P300 response.
ax.plot(epochs.times,
        target_epochs.get_data(picks=['Cz']).mean(axis=0).squeeze() * 1e6,
        # .get_data(picks=['Cz']) → shape (n_epochs, 1, n_times)
        # .mean(axis=0)           → shape (1, n_times) — average across epochs
        # .squeeze()              → shape (n_times,)   — drop the single-channel dimension
        # * 1e6                   → convert V → µV for a human-readable scale
        label='Target', color='red')
ax.plot(epochs.times,
        nontarget_epochs.get_data(picks=['Cz']).mean(axis=0).squeeze() * 1e6,
        label='Non-target', color='blue')
ax.axvline(0, color='black', linestyle='--', label='Stimulus onset')
ax.axhline(0, color='gray', linestyle=':')
ax.set_xlabel('Time (s)')
ax.set_ylabel('Amplitude (µV)')
ax.set_title(f'ERP at Cz — Subject {SUBJECT}, Session {SESSION} ({flash_ms} ms flash)')
ax.legend()
plt.tight_layout()
plt.show()

### Visualisation: Topographic Map of the P300

Scalp topography of the **target-average ERP** at 300, 400, and 500 ms. The P300 has a **central-parietal positive distribution** — expect a warm-coloured blob strongest over Pz, Cz, and CPz. A frontal or occipital distribution is atypical and warrants inspection.

In [ ]:
# Plot the average scalp voltage map (topography) for target epochs at three timepoints.
# times=[0.3, 0.4, 0.5] — snapshots at 300, 400, and 500 ms post-stimulus
# average=0.05 — each snapshot averages over a ±25 ms window around each time to reduce noise
# .average() computes the mean ERP across all target epochs before plotting
target_epochs.average().plot_topomap(times=[0.3, 0.4, 0.5], average=0.05)

## 10. Save epochs

Epochs are saved as a `.fif` file (MNE's native format). This file can be loaded directly in subsequent analysis notebooks with `mne.read_epochs()`.

In [ ]:
epochs.save(fif_path, overwrite=True)   # overwrite=True: safe to re-run — replaces any existing file for this subject/session
print(f"Saved: {fif_path}")

## Summary

This notebook produced a single `.fif` epochs file ready for classification. What was done:

1. **Loaded** the raw CSV with manually supplied column names (no header row in the dataset)
2. **Built** an MNE `RawArray` (µV → V, STIM channel added)
3. **Applied** standard_1005 montage
4. **Filtered:** notch at 50 Hz, then bandpass 0.1–30 Hz
5. **Epoched** −0.2 to 0.8 s around each stimulus onset, baseline-corrected to pre-stimulus window
6. **Rejected** epochs exceeding 100 µV peak-to-peak (no ICA)
7. **Aligned** target/non-target labels to surviving epochs
8. **Saved** epochs to `.fif`

**Next step:** Load the `.fif` in a classification notebook and train a P300 decoder (e.g. LDA on xDAWN-covariance features, or a Riemannian classifier).